In [1]:
# ==========================================================
# LIGHT NOVEL RECOMMENDATION SYSTEM
# PART 1
# Dataset + Cleaning + TF-IDF Model
# ==========================================================


import pandas as pd
import numpy as np
import random
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ==========================================================
# LOAD DATASET
# ==========================================================

df = pd.read_parquet("wn.parquet")

print("Dataset Shape:")
print(df.shape)


# ==========================================================
# HANDLE MISSING VALUES
# ==========================================================

required_columns = [
    "title",
    "authors",
    "genres",
    "tags",
    "language",
    "rating",
    "status_coo",
    "description"
]


for col in required_columns:

    if col in df.columns:
        df[col] = df[col].fillna("")


# ==========================================================
# REMOVE UNWANTED CONTENT
# ==========================================================

blocked = [
    "yaoi",
    "boys love",
    "Harem"
]


pattern = "|".join(blocked)


df = df[
    ~(
        df["genres"]
        .astype(str)
        .str.lower()
        .str.contains(pattern, na=False)
    )
]


df = df.reset_index(drop=True)


print("After filtering:")
print(df.shape)



# ==========================================================
# RATING CONVERSION
# ==========================================================

# ==========================================================
# FIX RATING COLUMN
# ==========================================================

def extract_rating(value):

    value = str(value)

    numbers = re.findall(
        r"\d+\.\d+|\d+",
        value
    )

    if len(numbers) > 0:
        rating = float(numbers[0])

        # keep only valid ratings
        if rating <= 5:
            return rating

    return 0



df["rating"] = df["rating"].apply(
    extract_rating
)


print(df["rating"].describe())



# ==========================================================
# TEXT CLEANING
# ==========================================================

def clean(text):

    text = str(text).lower()

    text = re.sub(
        r"[^\w\s]",
        " ",
        text
    )

    return text



# ==========================================================
# CREATE SEARCH TEXT
# ==========================================================


df["combined"] = (

    df["title"].astype(str)
    + " "
    +
    df["genres"].astype(str)
    + " "
    +
    df["tags"].astype(str)
    + " "
    +
    df["description"].astype(str)

)


df["combined"] = df["combined"].apply(clean)



# ==========================================================
# TF-IDF
# ==========================================================


vectorizer = TfidfVectorizer(

    stop_words="english",

    max_features=10000,

    ngram_range=(1,2)

)



tfidf_matrix = vectorizer.fit_transform(
    df["combined"]
)



print("\nTF-IDF Ready")

print(
    "Matrix Shape:",
    tfidf_matrix.shape
)

Dataset Shape:
(11770, 30)
After filtering:
(10143, 30)
count    10143.000000
mean         3.654994
std          0.617640
min          0.000000
25%          3.300000
50%          3.700000
75%          4.100000
max          5.000000
Name: rating, dtype: float64

TF-IDF Ready
Matrix Shape: (10143, 10000)


In [118]:
# ==========================================================
# LIGHT NOVEL RECOMMENDATION SYSTEM
# PART 2
# Recommendation Functions + Filters
# ==========================================================


# ==========================================================
# FORMAT TEXT
# ==========================================================

def clean_list(value):

    if isinstance(value, (list, np.ndarray)):

        return ", ".join(
            map(str, value)
        )

    return str(value)

def format_text(value):

    if pd.isna(value) or str(value).strip() == "":
        return "N/A"

    return str(value)



# ==========================================================
# GET RECOMMENDATIONS
# ==========================================================


def get_results(
        query,
        language="Any",
        minimum_rating=0,
        top_n=5,
        exclude_yaoi=True,
        exclude_bl=True,
        exclude_Harem=True
):


    query = clean(query)



    # Convert query into TF-IDF

    query_vector = vectorizer.transform(
        [query]
    )


    # Calculate similarity

    similarity = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()



    # Copy dataframe

    result = df.copy()


    result["similarity"] = similarity



    # -----------------------------
    # Language Filter
    # -----------------------------

    if language != "Any":

        result = result[
            result["language"]
            .astype(str)
            .str.lower()
            .str.contains(
                language.lower(),
                na=False
            )
        ]



    # -----------------------------
    # Rating Filter
    # -----------------------------

    if minimum_rating > 0:

        result = result[
            result["rating"] >= minimum_rating
        ]



    # -----------------------------
    # Genre Exclusion
    # -----------------------------


    banned = []


    if exclude_yaoi:
        banned.append("yaoi")


    if exclude_bl:
        banned.append("boys love")


    if exclude_Harem:
        banned.append("Harem")



    if banned:


        pattern = "|".join(banned)


        result = result[
            ~(
                result["genres"]
                .astype(str)
                .str.lower()
                .str.contains(
                    pattern,
                    na=False
                )
            )
        ]



    # Sort by similarity

    result = result.sort_values(
        by="similarity",
        ascending=False
    )


    return result.head(top_n)




# ==========================================================
# DISPLAY RECOMMENDATIONS
# ==========================================================


def recommend(
        query,
        language="Any",
        minimum_rating=0,
        top_n=5
):


    results = get_results(
        query,
        language,
        minimum_rating,
        top_n
    )



    if results.empty:

        return "❌ No novels found"



    output = ""

    output += "="*70+"\n"
    output += "📚 LIGHT NOVEL RECOMMENDATIONS\n"
    output += "="*70+"\n\n"



    for i, (_, row) in enumerate(
    results.iterrows(),
    1
):

        output += f"{i}. {format_text(row['title'])}\n"

        output += (
            f"⭐ Rating : {row['rating']:.2f}\n"
        )

        output += (
            f"👤 Author : {clean_list(row['authors'])}\n"
        )

        output += (
            f"📚 Genres : {clean_list(row['genres'])}\n"
        )

        output += (
            f"🏷 Tags : {clean_list(row['tags'])}\n"
        )
        output += (
            f"🎯 Similarity : "
            f"{row['similarity']*100:.2f}%\n"
        )


        output += "-"*70+"\n"


    return output





# ==========================================================
# RANDOM RECOMMENDATION
# ==========================================================


def random_recommend(
        language="Any",
        minimum_rating=0,
        top_n=5
):


    result = df.copy()



    if language != "Any":

        result = result[
            result["language"]
            .astype(str)
            .str.lower()
            .str.contains(
                language.lower(),
                na=False
            )
        ]



    result = result[
        result["rating"] >= minimum_rating
    ]



    if result.empty:

        return "❌ No novels available"



    sample = result.sample(
        min(top_n,len(result))
    )



    output = ""

    output += "🎲 RANDOM RECOMMENDATIONS\n"
    output += "="*70+"\n\n"



    for i,(_,row) in enumerate(
        sample.iterrows(),
        1
    ):

        output += (
            f"{i}. {row['title']}\n"
        )

        output += (
            f"⭐ Rating : {row['rating']}\n"
        )

        output += (
            f"📚 Genres : {row['genres']}\n"
        )

        output += "-"*70+"\n"



    return output

In [119]:
# ==========================================================
# LIGHT NOVEL RECOMMENDATION SYSTEM
# PART 3
# User Interface / Testing
# ==========================================================


# ==========================================================
# AVAILABLE LANGUAGES
# ==========================================================

languages = sorted(
    df["language"]
    .astype(str)
    .replace("", np.nan)
    .dropna()
    .unique()
)


languages = ["Any"] + list(languages)



# ==========================================================
# MAIN PROGRAM
# ==========================================================


print("="*70)
print("📚 LIGHT NOVEL RECOMMENDATION SYSTEM")
print("="*70)


while True:


    print("\nOptions:")
    print("1. Search Recommendation")
    print("2. Random Recommendation")
    print("3. Exit")


    choice = input(
        "\nEnter choice (1/2/3): "
    )



    # ------------------------------------------------------
    # SEARCH
    # ------------------------------------------------------

    if choice == "1":


        query = input(
            "\nSearch Novel: "
        )


        language = input(
            "Language (Enter for Any): "
        )


        if language.strip() == "":
            language = "Any"



        rating_input = input(
            "Minimum Rating (Enter for 0): "
        )


        if rating_input.strip() == "":
            minimum_rating = 0

        else:
            minimum_rating = float(
                rating_input
            )



        number_input = input(
            "Number of Results (Enter for 5): "
        )


        if number_input.strip() == "":
            top_n = 5

        else:
            top_n = int(
                number_input
            )



        print("\nSearching...\n")



        print(
            recommend(
                query,
                language,
                minimum_rating,
                top_n
            )
        )




    # ------------------------------------------------------
    # RANDOM
    # ------------------------------------------------------

    elif choice == "2":


        language = input(
            "Language (Enter for Any): "
        )


        if language.strip() == "":
            language = "Any"



        rating_input = input(
            "Minimum Rating (Enter for 0): "
        )


        if rating_input.strip() == "":
            minimum_rating = 0

        else:
            minimum_rating = float(
                rating_input
            )



        top_input = input(
            "Number of Novels (Enter for 5): "
        )


        if top_input.strip() == "":
            top_n = 5

        else:
            top_n = int(
                top_input
            )



        print(
            random_recommend(
                language,
                minimum_rating,
                top_n
            )
        )



    # ------------------------------------------------------
    # EXIT
    # ------------------------------------------------------

    elif choice == "3":

        print(
            "\nThank you for using Light Novel Recommender 📚"
        )

        break



    else:

        if choice.strip() != "":
            print("Invalid option!")

📚 LIGHT NOVEL RECOMMENDATION SYSTEM

Options:
1. Search Recommendation
2. Random Recommendation
3. Exit

Thank you for using Light Novel Recommender 📚
